# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading, exploring, and analyzing the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj/) using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

## Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Install the mlcroissant library if not already installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset with Croissant
dataset = mlc.Dataset(url)

# Access and print dataset metadata
meta = dataset.metadata
print(f"Dataset title:", meta.name)
print(f"Description:\n{meta.description}")

## 2. Data Overview
Review available record sets and their fields. All entities are referenced by their `@id` values for reproducibility and accurate referencing.

In [ ]:
# List all record set @id values defined in the dataset
print("Available record sets in the dataset:")
for rs in dataset.record_sets:
    print(f" - {rs['@id']}")

# For each record set, print available field @id values
print("\nFields in each record set:")
for rs in dataset.record_sets:
    print(f"\nRecord set '@id': {rs['@id']} - Name: {rs.get('name', '[no name]')}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]  # handle possible single field case
    for field in fields:
        if isinstance(field, dict):
            print(f"  - Field '@id': {field['@id']}  (name: {field.get('name','')})")
        else:
            print(f"  - Field: {field}")

## 3. Data Extraction
Load data from a specific record set into DataFrames for analysis.
All record sets and field references are by their `@id`.

In [ ]:
# Collect record set @id identifiers
record_sets_ids = [rs['@id'] for rs in dataset.record_sets]

# Extract data from each record set
dfs = {}
for record_set_id in record_sets_ids:
    recs = list(dataset.records(record_set=record_set_id))
    dfs[record_set_id] = pd.DataFrame(recs)

# Print columns of the first record set
main_rs_id = record_sets_ids[0]
print(f"Columns in main record set ({main_rs_id}):")
print(dfs[main_rs_id].columns.tolist())
dfs[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply typical preprocessing steps such as filtering, normalizing, and grouping by key demographic or assessment fields.

Below illustrates filtering for participants with PHQ-9 score (as an example numeric field) above a threshold, normalizing that column, and grouping by gender (as a demo categorical field). *Adjust field `@id`s from overview as needed depending on dataset details.*

In [ ]:
# --- REPLACE these with the actual @id values as per Overview cell output ---
# Example @id for PHQ-9 score and gender
numeric_field_id = None
group_field_id = None
# Inspect columns in main record set to find @id matches
print("Columns available:")
print(dfs[main_rs_id].columns.tolist())

# Assign the actual @id values below
# For illustration, let's guess some example @id values (these must be changed accordingly)
# For example:
# numeric_field_id = 'https://api.app.sen.science/frontiers/7160411/PHQ9_score_field_id'
# group_field_id = 'https://api.app.sen.science/frontiers/7160411/gender_field_id'

# Choose field @id's for analysis (use actual ones from your data):
for col in dfs[main_rs_id].columns:
    if 'phq' in col.lower():
        numeric_field_id = col
    if 'gend' in col.lower():
        group_field_id = col

if numeric_field_id is None or group_field_id is None:
    print("Warning: Could not auto-detect numeric_field_id/group_field_id. Please assign manually after inspecting available columns.")

# Proceed if fields are found
if numeric_field_id:
    threshold = 10
    df_main = dfs[main_rs_id]
    # Ensure numeric
    df_main[numeric_field_id] = pd.to_numeric(df_main[numeric_field_id], errors='coerce')
    filtered_df = df_main[df_main[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df[[numeric_field_id, group_field_id]].head())

    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame("mean").reset_index()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No numeric field selected for analysis.")

## 5. Visualization
Visualize data distributions or relationships between assessment scores and demographic fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Distribution of numeric field, and comparison across groups
if numeric_field_id and group_field_id:
    plt.figure(figsize=(10,4))
    sns.histplot(dfs[main_rs_id][numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # Boxplot by group
    plt.figure(figsize=(8,5))
    sns.boxplot(data=dfs[main_rs_id], x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=30)
    plt.show()
else:
    print("Visualization skipped: numeric or group field not found.")

## 6. Conclusion
In this notebook, we demonstrated loading the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj/) using the `mlcroissant` library, explored its structure based on Croissant `@id` references, extracted tabular data, and performed basic exploratory analysis and visualizations on key fields. For your own applications, further analyses (e.g., advanced modeling, longitudinal trend, or equity focus) can be pursued by building on this reproducible, standards-driven pipeline.